# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
before = len(df)

df = df.drop_duplicates().copy()

log(
    'duplicates',
    'Dropped exact duplicate rows',
    before - len(df)
)

[duplicates] Dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
df['price'] = (df['price'].astype(str)
               .str.replace('$', '', regex=False)
               .str.replace(',', '', regex=False)
               .str.strip()
               .astype(float))

log(
    'price',
    'Removed dollar signs and converted price to float',
    len(df)
)

[price] Removed dollar signs and converted price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

dropped_missing = df['qty'].isna().sum()
dropped_negative = (df['qty'] < 0).sum()

df = df[df['qty'].notna() & (df['qty'] > 0)].copy()
df['qty'] = df['qty'].astype(int)

log(
    'qty',
    'Dropped rows with missing or negative quantity',
    dropped_missing + dropped_negative
)

[qty] Dropped rows with missing or negative quantity (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())

before_item = df['item'].copy()

df['item'] = df['item'].str.strip().str.title()

ITEM_MAP = {
    'Cheese Burger': 'Cheeseburger'
}

df['item'] = df['item'].replace(ITEM_MAP)

rows_changed = (before_item != df['item']).sum()

log(
    'item',
    'Standardized item names into three products',
    rows_changed
)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] Standardized item names into three products (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
print(df['category'].value_counts())

before_category = df['category'].copy()

df['category'] = (df['category']
                  .str.strip()
                  .str.lower()
                  .str.replace('-', '', regex=False))

CATEGORY_MAP = {
    'apparel': 'merch',
    'raingear': 'raingear'
}

df['category'] = df['category'].replace(CATEGORY_MAP)

rows_changed = (before_category != df['category']).sum()

log(
    'category',
    'Normalized categories and rolled Apparel into Merch',
    rows_changed
)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] Normalized categories and rolled Apparel into Merch (231 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert sorted(df['item'].unique()) == [
    'Cheeseburger',
    'Foam Finger',
    'Rain Poncho'
]

assert sorted(df['category'].unique()) == [
    'food',
    'merch',
    'raingear'
]
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
df['revenue'] = df['qty'] * df['price']

revenue_by_category = (df.groupby('category')['revenue']
                       .sum()
                       .sort_values(ascending=False)
                       .round(2))

print(revenue_by_category)

print()
print(f"Overall revenue: ${df['revenue'].sum():.2f}")

category
food        1656.0
merch       1572.0
raingear    1512.0
Name: revenue, dtype: float64

Overall revenue: $4740.00


**What I would tell the vendor:** I would tell the vendor to stock more food items because Food generated the most revenue at $1,656.00.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,Dropped exact duplicate rows,15
1,price,Removed dollar signs and converted price to float,300
2,qty,Dropped rows with missing or negative quantity,25
3,item,Standardized item names into three products,126
4,category,Normalized categories and rolled Apparel into ...,231


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Dropping duplicates changed the revenue total the most. Revenue went from 4,852.50 to 4,594.50 after, a decrease of $258

b)I chose to combine Apparel with Merch. A reasonable alternative would be to keep them as separate categories. If they were separate, Merch would have 856.50 dollars in revenue and Apparel would have 715.50, instead of 1,572.00 combined under Merch. The overall revenue would still be 4,740.00. I combined them because I treated apparel as part of the broader merchandise category.